In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, load_iris, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.decomposition import PCA as SklearnPCA
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("="*80)
print("ALGORITHM 1: K-MEANS CLUSTERING")
print("="*80)

print("\n--- Approach 1: Standard K-Means from Scratch ---")

class KMeansScratch:
    def __init__(self, n_clusters=3, max_iters=100, tol=1e-4):
        self.n_clusters = n_clusters
        self.max_iters = max_iters
        self.tol = tol
        
    def fit(self, X):
        # Random initialization of centroids
        n_samples, n_features = X.shape
        random_indices = np.random.choice(n_samples, self.n_clusters, replace=False)
        self.centroids = X[random_indices]
        
        for _ in range(self.max_iters):
            # Assign clusters
            distances = self._compute_distances(X)
            self.labels = np.argmin(distances, axis=1)
            
            # Update centroids
            new_centroids = np.zeros_like(self.centroids)
            for k in range(self.n_clusters):
                if np.sum(self.labels == k) > 0:
                    new_centroids[k] = X[self.labels == k].mean(axis=0)
                else:
                    new_centroids[k] = self.centroids[k]
            
            # Check convergence
            if np.linalg.norm(new_centroids - self.centroids) < self.tol:
                break
                
            self.centroids = new_centroids
        
        return self
    
    def _compute_distances(self, X):
        distances = np.zeros((X.shape[0], self.n_clusters))
        for k in range(self.n_clusters):
            distances[:, k] = np.linalg.norm(X - self.centroids[k], axis=1)
        return distances
    
    def predict(self, X):
        distances = self._compute_distances(X)
        return np.argmin(distances, axis=1)

# Generate synthetic dataset
X_kmeans, y_kmeans = make_blobs(n_samples=300, n_features=2, centers=3, 
                                 cluster_std=0.60, random_state=42)

# Apply K-Means from scratch
kmeans_scratch = KMeansScratch(n_clusters=3)
kmeans_scratch.fit(X_kmeans)
predictions = kmeans_scratch.predict(X_kmeans)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X_kmeans[:, 0], X_kmeans[:, 1], c=y_kmeans, cmap='viridis', alpha=0.6)
axes[0].scatter(kmeans_scratch.centroids[:, 0], kmeans_scratch.centroids[:, 1], 
                c='red', marker='X', s=200, linewidth=3)
axes[0].set_title('True Labels with Centroids')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

axes[1].scatter(X_kmeans[:, 0], X_kmeans[:, 1], c=predictions, cmap='viridis', alpha=0.6)
axes[1].scatter(kmeans_scratch.centroids[:, 0], kmeans_scratch.centroids[:, 1], 
                c='red', marker='X', s=200, linewidth=3)
axes[1].set_title('K-Means Predictions (Scratch)')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
plt.tight_layout()
plt.show()

print("Dataset: Synthetic Blobs (300 samples, 2 features, 3 clusters)")
print(f"Centroids found: {kmeans_scratch.centroids}")